In [1]:
import requests
import pandas as pd
import io
import os

In [2]:
if os.path.exists('istat_raw.csv'):
    print("file già esistente, la chiamata API è stata skippata")
else:
    url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
    headers_csv = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

    r = requests.get(url, headers=headers_csv, timeout=120)
    print(r.status_code)

    if r.status_code == 200:
        with open('istat_raw.csv', 'w', encoding='utf-8') as f:
            f.write(r.text)
        print("Salvato su disco, non serve richiamare l'API di nuovo")

# l'API SDMX di ISTAT ha un limite di 5 chiamate al minuto: se lo superi,
# l'IP viene bloccato per 24-48 ore (un meccanismo un po' becero, a dirla
# come l'ha detto il prof)

# dato che i dati storici sugli incidenti non cambiano in tempo reale,
# non ha senso interrogare l'API ogni volta che eseguo il notebook.
# per questo aggiungo un controllo con if/else: se il file istat_raw.csv
# esiste già sul disco, salto del tutto la chiamata API; altrimenti,
# solo la prima volta, faccio la chiamata e appena arriva la risposta
# la salvo subito su disco. così, anche rieseguendo il notebook da capo
# più volte (es. con Restart & Run All), non rischio mai di superare il
# limite di chiamate al minuto, e mostro anche un po' di pietà per
# l'infrastruttura non esattamente performante di ISTAT

file già esistente, la chiamata API è stata skippata


In [3]:
df = pd.read_csv('istat_raw.csv')
print(df.info())
df

# visualizzo le informazioni del dataframe, ci sono 573.552 righe e 16 colonne ma 9 di queste
# sono completamente vuote (0 non-null):
# OBS_STATUS, NOTE_DS, NOTE_REF_AREA, NOTE_DATA_TYPE, NOTE_RESULT,
# NOTE_TIME_PERIOD, BASE_PER, UNIT_MEAS, UNIT_MULT
# quindi le rimuoverò in fase di pulizia

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573547,IT1:41_983(1.0),A,111107,ROADACC,9,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573548,IT1:41_983(1.0),A,111107,ROADACC,9,2021,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573549,IT1:41_983(1.0),A,111107,ROADACC,9,2022,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573550,IT1:41_983(1.0),A,111107,ROADACC,9,2023,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print(df['REF_AREA'].astype(str).str.len().unique())
print(df['DATA_TYPE'].unique())
print(df['RESULT'].unique())
print(df['DATAFLOW'].unique())
print(df['FREQ'].unique())

# ho controllato quante cifre hanno i codici REF_AREA convertendoli in stringa
# e misurandone la lunghezza: sono uscite lunghezze diverse (4, 5, 6) invece
# di un valore uniforme. Questo succede perché la colonna è stata letta da
# pandas come numero (int64), e i numeri non mantengono gli zeri davanti,
# quindi un codice come 001001 viene letto e salvato come 1001 (4 cifre).
# i codici comune ISTAT dovrebbero invece essere sempre a 6 cifre, quindi
# l'ho corretto

# ho controllato anche i valori unici di DATA_TYPE, RESULT, DATAFLOW e FREQ per capire
# cosa sono e cosa contengono queste colonne prima di
# usarle nell'analisi:
# - DATA_TYPE ha 2 valori: KILLINJ, ROADACC;
# - RESULT ha 3 valori: F, M, 9;
# - DATAFLOW e FREQ hanno un solo valore, rispettivamente 'IT1:41_983(1.0)' e 'A'
# da questi nomi non è ancora chiaro il significato esatto di alcuni valori (es. cosa
# rappresenta il valore 9), dopo controllerò, ma posso dire già con certezza che nè DATAFLOW nè FREQ
# hanno dei valori sensati perchè entrambi ne hanno solo uno per tutte le righe, quindi
# non apportano alcuna informazione utile

[4 5 6]
<StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
<StringArray>
['F', 'M', '9']
Length: 3, dtype: str
<StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str
<StringArray>
['A']
Length: 1, dtype: str


In [5]:
df['REF_AREA'] = df['REF_AREA'].astype(str).str.zfill(6)

print(df['REF_AREA'].head(10))
print(df['REF_AREA'].astype(str).str.len().unique())

# prima avevo trovato che REF_AREA aveva lunghezze diverse (4, 5, 6 cifre)
# perché pandas l'aveva letta come numero, perdendo gli zeri davanti ai
# codici comune. qui la converto in stringa con zfill(6), che aggiunge
# zeri a sinistra finché la stringa non arriva a 6 caratteri, così i
# codici tornano tutti uniformi e nel formato giusto e uguale per tutti i record

# controllo che il fix abbia funzionato stampando le prime righe e
# ricontrollo le lunghezze. il risultato [6] conferma che ora tutti
# i codici hanno 6 cifre

0    001001
1    001001
2    001001
3    001001
4    001001
5    001001
6    001001
7    001001
8    001001
9    001001
Name: REF_AREA, dtype: str
[6]


In [6]:
df[df['DATA_TYPE'] == 'ROADACC']['OBS_VALUE'].describe()

# guardando i numeri di .describe() si vede che qualcosa non è regolare:
# la mediana (50%) è 3, ma la media è quasi 25 (molto più alta della
# mediana). anche la deviazione standard (257) è enorme, circa 10 volte
# la media stessa. quando media e mediana sono così distanti, e la
# deviazione standard è più grande della media, di solito vuol dire che
# c'è almeno un valore molto più alto degli altri che tira su la media
# rispetto al resto dei dati, cioè un possibile outlier

count    191184.000000
mean         24.997599
std         257.519973
min           0.000000
25%           1.000000
50%           3.000000
75%          12.000000
max       23135.000000
Name: OBS_VALUE, dtype: float64

In [7]:
df[(df['DATA_TYPE'] == 'ROADACC') & (df['OBS_VALUE'] == 23135)]

# vado a vedere a quale riga appartiene il valore più alto (23135, il max
# uscito da .describe()), per capire se è un errore nei dati o un valore legittimo
# il codice REF_AREA che esce è 058091, che cercando su internet, corrisponde a Roma Capitale:
# essendo il comune più popoloso d'Italia, ha senso che abbia molti più
# incidenti rispetto agli altri comuni solo per via del traffico e delle dimensioni

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
348636,IT1:41_983(1.0),A,058091,ROADACC,9,2004,23135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df = df.drop(columns=['OBS_STATUS', 'NOTE_DS', 'NOTE_REF_AREA', 'NOTE_DATA_TYPE',
                       'NOTE_RESULT', 'NOTE_TIME_PERIOD', 'BASE_PER', 'UNIT_MEAS', 'UNIT_MULT', 'FREQ', 'DATAFLOW'])

# faccio pulizia togliendo queste colonne dal dataset, ovvero tutte quelle con NaN e le due
# colonne (DATAFLOW e FREQ) che non apportano alcuna informazione utile

In [9]:
df.info()

# controllo con .info() che il drop abbia funzionato: ora il dataframe
# ha 5 colonne invece di 16, tutte piene al 100% (573.552 valori
# non-null su ognuna), e la memoria occupata è scesa da 70 MB a 22 MB

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   REF_AREA     573552 non-null  str  
 1   DATA_TYPE    573552 non-null  str  
 2   RESULT       573552 non-null  str  
 3   TIME_PERIOD  573552 non-null  int64
 4   OBS_VALUE    573552 non-null  int64
dtypes: int64(2), str(3)
memory usage: 21.9 MB


In [10]:
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'F')]['OBS_VALUE'].describe())
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'M')]['OBS_VALUE'].describe())
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == '9')]['OBS_VALUE'].describe())

# faccio lo stesso controllo .describe() già fatto per ROADACC, ma
# questa volta sui feriti (RESULT=F), sui morti (RESULT=M) e sul 9 che non so ancora cosa sia:
# - F: media 35, mediana 5, max 30254 -> stessa forma fortemente
#   asimmetrica vista per ROADACC, pochi casi enormi tirano su la media
# - M: media 0.53, mediana 0 -> più della metà dei comuni/anni ha
#   zero morti, ha senso perché morire in un incidente è un evento raro.
#   il massimo però è 363, un numero alto, da verificare prima di darlo per buono
# - 9: dalla cella n° 7 del controllo sul massimo di Roma ho già visto che
#   RESULT=9 compare insieme a DATA_TYPE=ROADACC (REF_AREA=058091, TIME_PERIOD=2004).
#   ho anche dimostrato che il conteggio delle righe con DATA_TYPE=KILLINJ e RESULT=9 insieme
#   è zero (count=0), quindi quella combinazione non esiste mai. dato che DATA_TYPE ha solo
#   due valori possibili (KILLINJ, ROADACC, verificato con .unique()), e KILLINJ è escluso,
#   resta confermato che il 9 accompagna sempre e solo ROADACC, il che lo rende un placeholder. il 9
#   è lì solo per dire "l'evento è avvenuto" ma non tiene traccia del numero di morti e feriti in sè
# il perchè ho detto che F=feriti ed M=morti lo spiego più avanti

count    191184.000000
mean         35.091985
std         338.109881
min           0.000000
25%           1.000000
50%           5.000000
75%          18.000000
max       30254.000000
Name: OBS_VALUE, dtype: float64
count    191184.000000
mean          0.533225
std           2.783505
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         363.000000
Name: OBS_VALUE, dtype: float64
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: OBS_VALUE, dtype: float64


# Cercando online ho trovato che ISTAT usa convenzionalmente F per feriti
l'azienda OnData, che lavora a stretto contatto coi df dell'ISTAT, ha postato su GitHub
(fonte: https://github.com/ondata/guida-api-istat)
una guida su come giostrarsi tra i dati ISTAT e nel 3° esempio recita questo:

"Scarica solo i feriti negli incidenti stradali a Palermo, ultimi 10 record disponibili:

curl -kL -H "Accept: application/vnd.sdmx.data+csv;version=1.0.0" \
  "https://esploradati.istat.it/SDMXWS/rest/data/41_983/A.082053.KILLINJ.F?lastNObservations=10""

dove hanno applicato filtri dimensionali: A.082053.KILLINJ.F
- A = frequenza Annuale
- 082053 = codice comune Palermo
- KILLINJ = tipo dato (killed/injured)
- F = feriti
- Richiesto ultimi 10 record con lastNObservations=10 (in questo caso 10 anni perché frequenza è annuale)

per esclusione, visto che F=feriti e il 9 è un placeholder, M=morti


In [11]:
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'F') & (df['OBS_VALUE'] == 30254)])
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'M') & (df['OBS_VALUE'] == 363)])

# controllo a chi appartengono i due valori massimi trovati prima
# (30254 feriti, 363 morti): in entrambi i casi esce lo stesso comune,
# REF_AREA=058091 quindi Roma Capitale, lo steso identico comune che era
# già uscito come massimo per ROADACC. quindi è lo stesso fenomeno di
# prima: essendo il comune più popoloso e con più traffico d'Italia,
# ha senso che abbia i numeri più alti in assoluto su tutte le metriche

       REF_AREA DATA_TYPE RESULT  TIME_PERIOD  OBS_VALUE
348588   058091   KILLINJ      F         2004      30254
       REF_AREA DATA_TYPE RESULT  TIME_PERIOD  OBS_VALUE
348610   058091   KILLINJ      M         2002        363


In [12]:
df.groupby('TIME_PERIOD')['REF_AREA'].nunique()

# conto quanti comuni distinti ci sono per ogni anno, per vedere se la
# copertura è uniforme nel tempo o se ci sono buchi. dal 2001 al 2023 il
# calo è lento e graduale (da 8101 a 7901 comuni, circa -2.5% in 22 anni) -
# plausibile, è l'effetto delle fusioni di comuni che l'italia ha fatto
# negli ultimi decenni. ma dal 2023 al 2024 il calo è molto più brusco perchè va
# da 7901 a 6339 comuni, -1562 in un solo anno (-20%), troppo per essere
# spiegato dalle normali fusioni: qui serve un controllo più approfondito
# (che faccio nelle celle successive)

TIME_PERIOD
2001    8101
2002    8102
2003    8102
2004    8100
2005    8101
2006    8101
2007    8101
2008    8101
2009    8100
2010    8094
2011    8092
2012    8092
2013    8092
2014    8059
2015    8047
2016    8000
2017    7978
2018    7954
2019    7915
2020    7904
2021    7904
2022    7904
2023    7901
2024    6339
Name: REF_AREA, dtype: int64

In [13]:
print(df[df['TIME_PERIOD'] == 2023]['REF_AREA'].nunique())
print(df[df['TIME_PERIOD'] == 2024]['REF_AREA'].nunique())

# verifico se il calo visto sopra nel conteggio delle righe per anno
# riguarda intere righe di comuni mancanti, o solo singole metriche: conto
# i comuni distinti (REF_AREA) per il 2023 e il 2024. sono 7901
# comuni nel 2023, e solo 6339 nel 2024. una perdita di 1562 comuni interi,
# circa il 20%, come già notato nel conteggio totale delle righe.
# questo conferma che il 2024 ha dati probabilmente incompleti
# (mancano intere rilevazioni comunali), non solo "meno incidenti
# registrati" e per questo più avanti escluderò questo anno dal dataset princiale

7901
6339


In [14]:
df_completo = df.copy()
df = df_completo[df_completo['TIME_PERIOD'] != 2024].copy()

# tengo una copia con tutti gli anni (df_completo), incluso il 2024,
# si sa mai che dovesse servirmi, ed escludo il 2024 dal dataset principale (df).

# Ora che il dataset è pulito, procedo a fare la merge con SITUAS

In [15]:
df_pivot = df.pivot_table(index=['REF_AREA' , 'TIME_PERIOD'], columns=['DATA_TYPE' , 'RESULT'], values='OBS_VALUE')
df_pivot.head()

# trasformo il dataset con pivot_table:
# REF_AREA e TIME_PERIOD diventano l'indice (una riga per ogni comune+anno),
# DATA_TYPE e RESULT insieme diventano le nuove colonne, e OBS_VALUE i
# valori dentro quelle colonne

# dal .columns si vede che le colonne sono un MultiIndex, cioè identificate
# da coppie di valori e non da un nome singolo: ('KILLINJ', 'F'),
# ('KILLINJ', 'M'), ('ROADACC', '9'), sono le stesse tre combinazioni
# che avevo già trovato controllando i valori unici di DATA_TYPE e RESULT

DATA_TYPE            KILLINJ      ROADACC
RESULT                     F    M       9
REF_AREA TIME_PERIOD                     
001001   2001           10.0  0.0     5.0
         2002           10.0  0.0     5.0
         2003            7.0  0.0     4.0
         2004           13.0  0.0     9.0
         2005            2.0  0.0     2.0

In [16]:
df_pivot.columns = ['feriti', 'morti', 'incidenti']

# ho rinominayo le colonne del MultiIndex in nomi semplici e in italiano,
# rispettando lo stesso ordine visto sopra con .columns:
# ('KILLINJ', 'F') -> feriti, ('KILLINJ', 'M') -> morti,
# ('ROADACC', '9') -> incidenti

In [17]:
df_pivot.loc['058091']

# per sicurezza, conoscendo già i dati di Roma Capitale, l'ho cercata con .loc inserendo
# il codice ISTAT per vedere se effettivamente avessi fatto tutto bene

,feriti,morti,incidenti
TIME_PERIOD,,,
2001,27865.0,305.0,22220.0
2002,26696.0,363.0,21330.0
2003,26638.0,165.0,20426.0
2004,30254.0,260.0,23135.0
2005,28653.0,237.0,21902.0
2006,28209.0,231.0,21452.0
2007,26299.0,201.0,19960.0
2008,24062.0,190.0,18181.0
2009,24638.0,198.0,18561.0


In [18]:
df_pivot.isnull().sum()

# controllo se ci sono valori mancanti (NaN) nel pivot, su tutte le
# combinazioni comune+anno, non ci sono,
# quindi ogni comune/anno ha sempre un valore registrato per tutte le
# metriche (anche se a volte quel valore è 0, es. per i morti nei
# comuni più piccoli). posso quindi convertire le colonne in int senza
# preoccuparmi di buchi da gestire prima

feriti       0
morti        0
incidenti    0
dtype: int64

In [19]:
df_pivot = df_pivot.astype(int)

# convertto le colonne da float a int: non è strettamente necessario
# ma essendo conteggi (incidenti, feriti, morti)
# ha più senso rappresentarli come numeri interi, ed è anche
# più leggibile da guardare rispetto ad avere ".0" ovunque. il pivot li
# aveva trasformati in float per gestire eventuali NaN, ma avendo già
# verificato sopra che non ce ne sono, posso convertire senza perdere nulla

In [20]:
df_pivot = df_pivot.reset_index()
df_pivot.head()

# riporto REF_AREA e TIME_PERIOD a essere colonne normali invece che
# indice del dataframe (così come erano impostate dal pivot_table),
# per prepararmi al join con i dati SITUAS più avanti, che funziona
# meglio con colonne standard piuttosto che con l'indice a due livelli.
# al loro posto pandas crea un nuovo indice numerico sequenziale

,REF_AREA,TIME_PERIOD,feriti,morti,incidenti
0,001001,2001,10,0,5
1,001001,2002,10,0,5
2,001001,2003,7,0,4
3,001001,2004,13,0,9
4,001001,2005,2,0,2


## Verifico che un solo anno di popolazione sia o meno una semplificazione accettabile

Il README specifica che i dati SITUAS sono statici e che è accettabile scaricarli una sola volta ("it's fine to download it manually once"), ma avvisa anche che il csv è specifico per anno, il dataset degli incidenti invece copre 23 anni (2001-2023).

Prima di scegliere se scaricare file CSV per un solo anno o per tutti gli anni, verifico quanto cambia realmente la popolazione di un comune nel tempo: se dal 2001, col CSV più vecchio, fino al 2023, con quello più recente, non vedo chissà che differenze allora la variazione è piccola (e quindi un solo anno è una semplificazione ragionevole, coerente con quanto suggerito dal README) se invece cambia abbastanza dovrò valutare se richiedere un dato per ogni anno. Analizzo i dataset e faccio le prove che occorrono

In [21]:
pop_2001 = pd.read_csv('Comuni - Dimensione Data Indagine 31-12-2001.csv', sep=';')
pop_2001

,Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,1,1,1001,1001,Agliè,NaN,TO,0,0,2574.0,2001,"13,1462",2001,2557,2001
1,1,1,1,1002,1002,Airasca,NaN,TO,0,0,3554.0,2001,"15,7393",2001,3543,2001
2,1,1,1,1003,1003,Ala di Stura,NaN,TO,0,0,479.0,2001,"46,3315",2001,480,2001
3,1,1,1,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1696.0,2001,"11,7315",2001,1687,2001
4,1,1,1,1005,1005,Alice Superiore,NaN,TO,0,0,616.0,2001,"7,3796",2001,619,2001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8097,1,1,103,103073,103073,Viganella,NaN,VB,0,0,204.0,2001,"13,6649",2001,204,2001
8098,1,1,103,103074,103074,Vignone,NaN,VB,0,0,1090.0,2001,"3,3806",2001,1087,2001
8099,1,1,103,103075,103075,Villadossola,NaN,VB,0,0,6908.0,2001,"18,7333",2001,6898,2001
8100,1,1,103,103076,103076,Villette,NaN,VB,0,0,244.0,2001,"7,3834",2001,241,2001


In [22]:
pop_2023 = pd.read_csv('Comuni - Dimensione Data Indagine 31-12-2023.csv', sep=';')
pop_2023

# carico i due csv di SITUAS, popolazione 2001 e 2023. uso sep=';' perché
# questi file, a differenza di istat_raw.csv, usano il punto e virgola
# come separatore invece della virgola

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,1,1,201,1001,1001,Agliè,NaN,TO,0,0,2562,2021,"13,1463",2023,2596,2023
1,1,1,1,201,1002,1002,Airasca,NaN,TO,0,0,3660,2021,"15,7393",2023,3686,2023
2,1,1,1,201,1003,1003,Ala di Stura,NaN,TO,0,0,467,2021,"46,3316",2023,472,2023
3,1,1,1,201,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1637,2021,"11,7397",2023,1617,2023
4,1,1,1,201,1006,1006,Almese,NaN,TO,0,0,6331,2021,"17,8741",2023,6315,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7895,5,20,111,111,111103,111103,Villaputzu,NaN,SU,0,0,4509,2021,"181,3947",2023,4438,2023
7896,5,20,111,111,111104,111104,Villasalto,NaN,SU,0,0,989,2021,"130,3596",2023,938,2023
7897,5,20,111,111,111105,111105,Villasimius,NaN,SU,0,0,3705,2021,"58,1781",2023,3721,2023
7898,5,20,111,111,111106,111106,Villasor,NaN,SU,0,0,6599,2021,"86,8137",2023,6600,2023


In [23]:
pop_2001 = pop_2001[['Codice Comune (alfanumerico)', 'Comune', 'Sigla automobilistica',
                      'Codice Provincia/Uts', 'Superficie (Kmq)', 'Popolazione residente']]
pop_2023 = pop_2023[['Codice Comune (alfanumerico)', 'Comune', 'Sigla automobilistica',
                      'Codice Provincia/Uts', 'Superficie (Kmq)', 'Popolazione residente']]

# tengo solo le colonne che mi servono da entrambi i file: il codice comune,
# il nome del comune, la sigla automobilistica, il codice provincia, la
# superficie in kmq (necessaria per calcolare la densità di incidenti per
# km² richiesta dal README) e la popolazione residente (necessaria per il
# calcolo pro capite). nel file 2023 c'è anche una colonna
# "Codice Provincia (Storico)", che non prendo perché mi serve quello
# attuale (Codice Provincia/Uts)

In [24]:
pop_2001

,Codice Comune (alfanumerico),Comune,Sigla automobilistica,Codice Provincia/Uts,Superficie (Kmq),Popolazione residente
0,1001,Agliè,TO,1,"13,1462",2557
1,1002,Airasca,TO,1,"15,7393",3543
2,1003,Ala di Stura,TO,1,"46,3315",480
3,1004,Albiano d'Ivrea,TO,1,"11,7315",1687
4,1005,Alice Superiore,TO,1,"7,3796",619
...,...,...,...,...,...,...
8097,103073,Viganella,VB,103,"13,6649",204
8098,103074,Vignone,VB,103,"3,3806",1087
8099,103075,Villadossola,VB,103,"18,7333",6898
8100,103076,Villette,VB,103,"7,3834",241


In [25]:
pop_2023

# ho verificato il salvataggio corretto dei cambiamenti delle colonne

,Codice Comune (alfanumerico),Comune,Sigla automobilistica,Codice Provincia/Uts,Superficie (Kmq),Popolazione residente
0,1001,Agliè,TO,201,"13,1463",2596
1,1002,Airasca,TO,201,"15,7393",3686
2,1003,Ala di Stura,TO,201,"46,3316",472
3,1004,Albiano d'Ivrea,TO,201,"11,7397",1617
4,1006,Almese,TO,201,"17,8741",6315
...,...,...,...,...,...,...
7895,111103,Villaputzu,SU,111,"181,3947",4438
7896,111104,Villasalto,SU,111,"130,3596",938
7897,111105,Villasimius,SU,111,"58,1781",3721
7898,111106,Villasor,SU,111,"86,8137",6600


In [26]:
pop_2001 = pop_2001.rename(columns={'Popolazione residente': 'pop_2001'})
pop_2023 = pop_2023.rename(columns={'Popolazione residente': 'pop_2023'})

# rinomino la colonna popolazione in ognuno dei due dataframe, perché
# altrimenti dopo il merge avrei due colonne con lo stesso nome
# (Popolazione residente) e non potrei distinguerle